
# 🎙️ **Individual Model Baseline — Ranking of 20 Audio Deepfake Detectors**

This notebook is part of the article **“DCMC-Fusion: Data-Centric and Model-Centric Ensemble Learning for Audio Deepfake Detection”**, by **Dora Ballesteros and Carlos Guerra**.

Its purpose is to evaluate and rank the performance of **20 individual audio deepfake detectors**, corresponding to five deep-learning architectures (DenseNet121, ResNet50, InceptionV3, ResNet34, and VGG16), each trained using four spectral representations (Mel, LOG, DWT, and CQT). All models are evaluated independently on the **same fixed held-out test partition** used in the fusion experiments. Their performance is assessed using Accuracy, Precision-macro, Recall-macro, and F1-macro, with the latter serving as the ranking criterion. The resulting ranking establishes an **individual-model baseline** for comparison with DCMC-Fusion and other fusion-based methods.



### 📥 Input Files

This experiment requires **two input files**, which must be available before executing this notebook:

1. **`inference_probabilities.csv`:** generated **prior to and independently of the DCMC-Fusion method**. It contains the inference probabilities produced by 20 previously trained models, corresponding to five architectures (ResNet50, DenseNet121, InceptionV3, ResNet34, and VGG16) and four spectral representations (Mel, LOG, DWT, and CQT). Each row represents an audio sample, and the `label` column contains its ground-truth class (0 = natural, 1 = synthetic). The synthetic-class scores are expressed on a scale from 0 to 100.

2. **`experiment_split.json`:** contains the predefined training and test indices, the original audio and label ordering, and the 20 model feature names. This file ensures that all individual models are evaluated using the **same fixed held-out test partition** employed in the fusion experiments.

### 📤 Output Files

After executing the notebook, the following CSV file is generated:

1. **`individual_model_baseline_ranking.csv`:** contains the ranking of the 20 individual audio deepfake detectors, sorted in descending order of **F1-macro**. For each model, the file reports its architecture, spectral representation, model identifier, and four global classification metrics (Accuracy, Precision-macro, Recall-macro, and F1-macro).

The ranking identifies the **best-performing individual model** on the held-out test partition, providing a reference baseline for comparison with DCMC-Fusion and other fusion-based methods.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

ROOT = Path('/content')  # Google Colab; change if running locally
CSV_PATH = ROOT / 'inference_probabilities.csv'
SPLIT_PATH = ROOT / 'experiment_split.json'
OUTPUT_PATH = ROOT / 'individual_model_baseline_ranking.csv'
THRESHOLD = 50.0  # The CSV stores class-1 probabilities on a 0–100 scale


def main():
    df = pd.read_csv(CSV_PATH)
    with SPLIT_PATH.open('r', encoding='utf-8') as file:
        split = json.load(file)

    # The split stores row positions, so original CSV order is essential.
    assert df['audio'].astype(str).tolist() == split['audio_order'], (
        'Audio order differs from experiment_split.json.'
    )
    assert df['label'].astype(int).tolist() == split['label_order'], (
        'Label order differs from experiment_split.json.'
    )

    test_indices = np.asarray(split['test_indices'], dtype=int)
    train_indices = np.asarray(split['train_indices'], dtype=int)
    assert len(np.unique(test_indices)) == len(test_indices)
    assert not np.intersect1d(train_indices, test_indices).size
    assert len(train_indices) + len(test_indices) == len(df)
    assert test_indices.min() >= 0 and test_indices.max() < len(df)

    test_df = df.iloc[test_indices]
    y_true = test_df['label'].astype(int).to_numpy()
    model_columns = split['feature_columns']
    assert len(model_columns) == 20
    assert all(col in test_df.columns for col in model_columns)

    results = []
    for column in model_columns:
        probabilities = test_df[column].to_numpy(dtype=float)
        assert np.isfinite(probabilities).all(), f'Invalid probabilities in {column}'
        assert ((probabilities >= 0) & (probabilities <= 100)).all(), column
        y_pred = (probabilities >= THRESHOLD).astype(int)
        architecture, representation = column.rsplit('_', 1)
        results.append({
            'architecture': architecture,
            'representation': representation.upper(),
            'model': column,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        })

    ranking = (pd.DataFrame(results)
               .sort_values('f1_macro', ascending=False)
               .reset_index(drop=True))
    ranking.index += 1
    ranking.index.name = 'rank'
    ranking.to_csv(OUTPUT_PATH, float_format='%.8f')

    print(f'Total samples: {len(df)}')
    print(f'Test samples: {len(test_df)} ({100 * len(test_df) / len(df):.2f}%)')
    print(f'Individual models: {len(ranking)}')
    print('\nRanking by F1-macro:')
    print(ranking.to_string(float_format=lambda x: f'{x:.6f}'))
    best = ranking.iloc[0]
    print(f'\nBest baseline: {best["model"]} | F1-macro = {best["f1_macro"]:.6f}')
    print(f'Ranking saved to: {OUTPUT_PATH}')


if __name__ == '__main__':
    main()


Total samples: 2484
Test samples: 746 (30.03%)
Individual models: 20

Ranking by F1-macro:
      architecture representation             model  accuracy  precision_macro  recall_macro  f1_macro
rank                                                                                                  
1            vgg16            LOG         vgg16_log  0.697051         0.770122      0.697051  0.675077
2         resnet50            LOG      resnet50_log  0.668901         0.676419      0.668901  0.665335
3     inception_v3            LOG  inception_v3_log  0.648794         0.649412      0.648794  0.648430
4      densenet121            LOG   densenet121_log  0.647453         0.647454      0.647453  0.647452
5     inception_v3            MEL  inception_v3_mel  0.611260         0.624589      0.611260  0.600578
6      densenet121            CQT   densenet121_cqt  0.553619         0.561981      0.553619  0.538039
7            vgg16            MEL         vgg16_mel  0.573727         0.613056      0